# 第5回 正則化による多重共線性の問題への対処


---


## 前回からの接続


第4回では、住宅の査定担当者として、11本の列で価格の式を立て、係数を読みました。そして、車庫の台数と車庫の面積のような似た列の組があると、係数の分け前が決まらず、符号さえ揺れることを見ました。

でも、データには80本の列があります。列を増やせば、もっと当たるかもしれません。そのかわり、似た列の組はもっと増えます。今日は、列を増やしても係数が暴れないように、係数の大きさを抑える方法を学びます。


---


## 今日の分析目標


あなたは前回の査定担当者です。列を増やして当たりを良くしながら、お客さんに説明できる係数を保つ。

数の列を35本すべて使って式を立て、係数がどう崩れるかを確かめます。そのうえで、係数の大きさに罰金をかける正則化（RidgeとLasso）で係数を抑え、罰金の強さを交差検証（cross-validation）で選びます。道具は、付いている説明書き（ドキュメント）を読んでから使います。最後に、説明に使える係数で査定します。もう1つのねらいは、初めての道具でも、説明書き（ドキュメント）で引数を読んでから使えるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、前回と同じ住宅の査定担当者です。家の条件から、売れる値段を見積もります。お客さんには、値段の理由を条件ごとに説明します。

決めたいのは、列を増やして見積もりの外れを小さくしながら、条件ごとの値段を係数で説明することです。ただ、データには80本の列があり、列を増やすほど、車庫の台数と車庫の面積のような似た列の組も増えます。


### 手元のデータ

前回と同じ、米国エイムズ市の資産評価事務所が記録した、2006〜2010年の住宅の売買です（De Cock, 2011）。1行が1軒の売買で、答えの列`SalePrice`は売れた価格（米ドル）です。前回と同じく、ふつうの売買で、居住面積が広すぎない家の2,412軒に絞って使います。


### 最後に出す答え

数の列をすべて使って外れを小さくした価格の式と、その式で言える「居住面積1㎡あたり何ドル、品質1段階あたり何ドル」という説明です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 足すとほかの列になる列の組（完全な多重共線性）があると、係数が一つに決まらないことを確かめられる
- Ridgeが「係数の二乗の和」に罰金をかけ、似た列の組の係数を近づけながら縮めることを説明できる
- Lassoが「係数の絶対値の和」に罰金をかけ、いくつかの係数をちょうど0にすることを確かめられる
- 関数のドキュメントを読んで、引数（`alpha`・`max_iter`・`alphas`など）の意味を確かめてから使える
- 罰金の強さαを、交差検証で選べる
- 正則化（regularization）した式の係数を、元の単位に戻して説明に使える


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の解析と、③ の解釈が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 列を増やす：もっと当たるが、係数は？


まず、数の列をすべて入れると、当たりと係数がどう変わるかを確かめます。前回と同じく、ふつうの売買で、居住面積が約372㎡以下の家を使います。今日は、数の列をすべて使います。ただし`MS SubClass`（建物の種類の番号）は、数の顔をした区分（第1回）なので外します。面積の列は、すべて㎡に直します。少しずつ、4つのセルに分けて進めます。


### お手本①：読み込んで、数の列を選ぶ

- 役割：データを読み込んでふつうの売買に絞り、数の列の名前を取り出す
- 入力：データの置き場所とファイル名
- 出力：表`df`と、数の列の名前`cols`

`select_dtypes('number')`は、数の列だけを選ぶメソッドです。後ろの`.columns`で、その列の名前を取り出します。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/ames_housing.csv')
normal = raw['Sale Condition'] == 'Normal'
df = raw[normal & (raw['Gr Liv Area'] <= 4000)].reset_index(drop=True)
cols = df.select_dtypes('number').columns     # 数の列の名前
print(len(df), '軒 /', len(cols), '本の数の列')


数の列は37本あります。ここから、当てたい価格（`SalePrice`）と、`MS SubClass`（建物の種類の番号。数の顔をした区分です。第1回）の2本を外します。そのために、`for`と`if`を1行に書くリスト内包表記を使います。

### おさらい：for文

```python
for i in range(3):
    print(i)          # 0, 1, 2 と表示される
```

並びの中身を1つずつ名前に入れながら、字下げした行を繰り返す書き方でした（第2回）。

### おさらい：if文

```python
for i in range(3):
    if i > 0:
        print(i)      # 1, 2 と表示される
```

`if`は、条件が成り立つときだけ、下の字下げした行を実行する書き方でした（第2回）。


### リスト内包表記：forを1行で書いて、リストを作る

`[式 for 名前 in 並び]`と書くと、並びの中身を1つずつ名前に入れて式を計算し、その結果を並べた新しいリストができます。後ろに`if 条件`を付けると、条件が成り立つものだけが残ります。第4回で、列を選ぶのに使った形です。


In [ ]:
print([i * 10 for i in range(3)])            # [0, 10, 20]
print([i for i in range(3) if i > 0])        # [1, 2]


### お手本②：2本を外して、35本にする

`c not in drop`は、`c`が`drop`の中に入っていないときTrueになります。


In [ ]:
drop = ('SalePrice', 'MS SubClass')
num = [c for c in cols if c not in drop]      # 2本を外す
print(len(num), '本の列')


### 文字列のin：言葉が入っているか

`in`は、文字列にも使えます。`'SF' in c`は、列の名前`c`に`SF`という文字が入っているときTrueです。


In [ ]:
print('SF' in 'Total Bsmt SF')               # True
print('SF' in 'Lot Area')                    # False


### お手本③：面積の列を、㎡に直す

面積の列は、名前に`SF`（平方フィート）・`Area`・`Porch`のどれかが入っています。`or`は「どれか1つでも成り立てば」です。


In [ ]:
for c in num:
    if 'SF' in c or 'Area' in c or 'Porch' in c:   # 面積の列
        df[c] = df[c] * 0.0929                      # → ㎡


### お手本④：欠けた値を補って、式を立てる

数の列には、欠けた値（例えば、道路に面した長さ`Lot Frontage`）があります。`SimpleImputer(strategy='median')`は、欠けた値を補う道具です（第3回）。訓練で`fit`すると各列の中央値を覚え、`transform`で欠けたところをその中央値で埋めます。


In [ ]:
tr, te = train_test_split(df, test_size=0.2, random_state=42)
y_tr, y_te = tr['SalePrice'].values, te['SalePrice'].values
imp = SimpleImputer(strategy='median').fit(tr[num])   # 中央値で補う
X_tr, X_te = imp.transform(tr[num]), imp.transform(te[num])
ols = LinearRegression().fit(X_tr, y_tr)


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが関数の名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。第3・4回の外れ幅の関数を、もう一度作ります。


### 変えてみる：外れ幅で、11本の式と比べる


In [ ]:
def mae(y, pred):      # 外れ幅の平均（MAE）
    return round(np.abs(y - pred).mean())

feat11 = ['Overall Qual', 'Gr Liv Area', 'Total Bsmt SF',
          'Garage Cars', 'Garage Area', 'Year Built',
          'Year Remod/Add', 'Full Bath', 'TotRms AbvGrd',
          'Fireplaces', 'Lot Area']
m11 = LinearRegression().fit(tr[feat11], y_tr)
print('11本の式:', mae(y_te, m11.predict(te[feat11])), 'ドル')
print('35本の式:', mae(y_te, ols.predict(X_te)), 'ドル')


35本すべてを使うと、外れ幅は19,061ドルから16,970ドルへ、2千ドル以上縮みました。列を増やすと、よく当たるようになります。では、係数はどうでしょうか。居住面積に関係する列を見てみます。


In [ ]:
coef_ols = pd.Series(ols.coef_, index=num)
print(coef_ols[['Gr Liv Area', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF']].round(1))


居住面積（`Gr Liv Area`）は1㎡あたり435ドル、1階の面積（`1st Flr SF`）が273ドル、2階の面積（`2nd Flr SF`）が248ドル、仕上げの低い部分の面積（`Low Qual Fin SF`）が−86ドルです。前回の「居住面積1㎡あたり約677ドル」が、4つの列に分かれ、一つずつでは読めない形になりました（足しても677にはなりません）。なぜでしょうか。


### TODO①：足すと、居住面積になる？

1階の面積・2階の面積・仕上げの低い部分の面積を足すと、居住面積とぴったり同じになるかを、`np.allclose`で確かめてください。`np.allclose(a, b)`は、2つの並びがどこも（計算の誤差を除いて）同じなら`True`を返します。


In [ ]:
# TODO: 1階 + 2階 + 仕上げの低い部分 が、居住面積と同じかを確かめる
# ヒント: np.allclose(df['Gr Liv Area'], df['1st Flr SF'] + … )
...


Trueになります。居住面積は、1階・2階・仕上げの低い部分の足し算そのものでした。地下室の面積（`Total Bsmt SF`）も、同じように3つの列の足し算です。

居住面積が「1階＋2階＋低い部分」とぴったり等しいなら、1階1㎡の効き目を「居住面積の係数」で表しても、「1階の係数」で表しても、合計さえ同じなら予測はまったく同じです。分け前の決め方は、無数にあるのです。第4回の車庫の組（相関0.88）は「ほぼ重なる」でしたが、これは完全に重なる場合です（完全な多重共線性）。435ドルや273ドルという一つずつの値には、意味がありません。

読むなら、組み合わせて読みます。1階が1㎡広いと、居住面積も1㎡広くなるので、1階の1㎡の効き目は435＋273＝約708ドル。2階の1㎡なら435＋248＝約683ドルです。


---


## 2. Ridge：係数の二乗に罰金をかける


1節では、列を増やすと当たりは良くなりましたが、係数はばらばらで、一つずつでは読めなくなりました。では、係数の大きさを抑えるにはどうすればよいでしょうか。

線形回帰（linear regression）は、「外れの二乗の合計」をできるだけ小さくする係数を選びました（第2回）。Ridge（リッジ回帰）は、そこに「係数の二乗の合計 × α」という罰金を足して、その合計を小さくする係数を選びます。

外れの二乗の合計 ＋ α ×（係数の二乗の合計）

罰金があるので、係数はむやみに大きくなれません。αが罰金の強さで、αを大きくするほど、係数は全体として0に向かって縮みます（一つずつの係数は、あとで見るように大きくなることもあります）。係数の大きさを公平に罰するため、先に標準化（standardization）しておきます（第3回。係数は「1標準偏差あたり」になります）。


In [ ]:
sc = StandardScaler().fit(X_tr)        # 物差しは、訓練だけで決める
Z_tr, Z_te = sc.transform(X_tr), sc.transform(X_te)


### お手本：ドキュメントで、引数を読む

- 役割：`Ridge`の説明書き（ドキュメント）から、引数`alpha`の説明を取り出して読む
- 入力：`Ridge`
- 出力：`alpha`の説明の数行

ライブラリの関数には、使い方の説明書きが付いています。`help(Ridge)`で全部を表示できますが、長いので、ここでは説明書きの文字列`Ridge.__doc__`から、`alpha`の説明のところだけを取り出します。`find`は、文字列の中で言葉が最初に出てくる位置（先頭から何文字目か）を返すメソッドです。`doc[i:i + 500]`は、`i`文字目から500文字ぶんを切り出す書き方です（リストの`[:3]`と同じ、スライスです）。


In [ ]:
doc = Ridge.__doc__                     # Ridge の説明書き（英語）
i = doc.find('alpha :')                 # alpha の説明が始まる位置
print(doc[i:i + 500])


`alpha`は「L2の項（係数の二乗の合計）に掛ける定数で、正則化の強さを決める（controlling regularization strength）」「0以上の数でなければならない」と書かれています。道具の引数は、名前から想像せず、説明書きで確かめてから使いましょう。英語の説明書きは、翻訳の道具を使って読んでもかまいません。


### TODO②：αを変えて、Ridgeの係数と外れ幅を見る

`for`文でαを1, 10, 100, 1000と変えて`Ridge(alpha=a)`を作り、テストでの外れ幅と、車庫の台数・車庫の面積の係数を表示してください。


In [ ]:
# TODO: for a in [1, 10, 100, 1000]: で Ridge を作り、
#       外れ幅と、車庫の台数・車庫の面積の係数を表示する
# ヒント: r = Ridge(alpha=a).fit(Z_tr, y_tr)
#         c = pd.Series(r.coef_, index=num)
...


αを大きくすると、車庫の台数と車庫の面積の係数は1,477と2,031（α=1）から、3,601と3,994（α=1000）へと、近づいていきます。外れ幅は、α=100で16,744ドルと、正則化なし（16,970ドル）より少し良くなり、α=1000では17,212ドルに悪くなりました。

でも、おかしくないでしょうか。「αが大きいほど縮む」はずなのに、車庫の2つの係数は、台数が約2.4倍、面積が約2倍に大きくなっています。大きな係数と並べて確かめます。


In [ ]:
cols5 = ['Gr Liv Area', 'Overall Qual', 'Year Built',
         'Garage Cars', 'Garage Area']
r1 = Ridge(alpha=1).fit(Z_tr, y_tr)
r1000 = Ridge(alpha=1000).fit(Z_tr, y_tr)
both = pd.DataFrame({'α=1': r1.coef_, 'α=1000': r1000.coef_},
                    index=num)
print(both.loc[cols5].round(0))


居住面積は17,739 → 10,019、品質は17,850 → 12,357、建築年は8,138 → 4,287と、大きな係数は大きく縮みました。Ridgeが縮めるのは、係数全体の大きさです。

車庫の広さは、品質・新しさ・居住面積と情報が重なっています（相関は0.5前後）。大きな係数が縮んだぶん、その重なる情報の分け前が、車庫の組に回ったのです。

では、車庫の台数と面積は、なぜ近づくのでしょうか。同じ情報を持つ2列なら、二つの係数の合計が同じとき、予測も同じです。そのとき、罰金の「係数の二乗の合計」がいちばん小さくなるのは、二つを半分ずつにしたときです（例えば、4と0なら4²＋0²＝16、2と2なら2²＋2²＝8）。車庫の2列は完全に同じではありませんが（相関0.88）、Ridgeは、似た列の組に、効き目をなるべく等しく分けようとするのです。


In [ ]:
# @title 図：α を変えたときの係数の動き
alphas = np.logspace(-1, 4, 30)
path = pd.DataFrame([Ridge(alpha=a).fit(Z_tr, y_tr).coef_ for a in alphas],
                    index=alphas, columns=num)
fig, ax = plt.subplots(figsize=(9, 4))
for c, col in [('Gr Liv Area', '#0066cc'), ('1st Flr SF', '#2a9d8f'), ('2nd Flr SF', '#8ab17d'),
               ('Garage Cars', '#e76f51'), ('Garage Area', '#e63946')]:
    ax.plot(path.index, path[c], label=c, color=col, lw=2)
ax.set_xscale('log'); ax.axhline(0, color='#555', lw=1)
ax.set_xlabel('α（罰金の強さ）'); ax.set_ylabel('係数（1標準偏差あたりのドル）')
ax.legend(fontsize=10); plt.tight_layout(); plt.show()


左のαが小さいところでは、居住面積・1階・2階の係数がばらばらですが、αを大きくすると、3本とも0に向かって縮んでいきます（近づき方は一様ではなく、2階は途中から1階と離れます）。車庫の台数と面積は、いったん大きくなりながら、近づいていきます。


---


## 3. Lasso：いくつかの係数を、ちょうど0にする


Ridgeは係数を縮め、似た列の組を近づけました。でも、ちょうど0になる係数はなく、35本の列はすべて式に残ったままです。では、係数を0にして、列を式から外せないでしょうか。

Lasso（ラッソ回帰）は、罰金を「係数の絶対値の合計 × α」にします。

外れの二乗の合計 ＋ α ×（係数の絶対値の合計）

小さな違いに見えますが、効き方はまったく違います。Lassoは、あまり役に立たない列の係数を、ちょうど0にします。0になった列は、式から外れたのと同じです。Lassoは、係数を縮めながら、列を選ぶのです。

一つ注意があります。scikit-learnの`Lasso`は、正確には「外れの二乗の合計 ÷（2 × 件数）＋ α ×（係数の絶対値の合計）」を小さくします。外れの側を、`Ridge`にはない「2 × 件数」で割っているのです。そのため、同じαの値でも、RidgeとLassoでは罰金の強さの物差しが違い、αの大きさどうしは比べられません。


### おさらい：ドキュメントで、引数を読む

```python
doc = Ridge.__doc__
i = doc.find('alpha :')
print(doc[i:i + 500])
```

説明書きの文字列から、読みたい引数のところを`find`で探して取り出します。


### 変えてみる：Lassoのmax_iterを読む

Lassoは、係数を少しずつ動かしながら答えに近づく計算のしかたをします。その繰り返しの回数の上限が、引数`max_iter`です。`Ridge`を`Lasso`に、`'alpha :'`を`'max_iter :'`に変えて読みます。


In [ ]:
doc = Lasso.__doc__
i = doc.find('max_iter :')
print(doc[i:i + 300])


「繰り返しの回数の上限（The maximum number of iterations）」で、初めから決まっている値は1000回です。今日のように列が多いと、1000回では答えにたどり着かないことがあるので、`max_iter=100000`と大きくして使います（足りないと、注意の表示が出ます）。


### TODO③：αを変えて、Lassoで0になる列を数える

`for`文でαを10, 100, 300, 1000, 3000と変えて`Lasso(alpha=a, max_iter=100000)`を作り、テストでの外れ幅と、係数がちょうど0になった列の数を表示してください。0になった列の数は、`(l.coef_ == 0).sum()`で数えられます（Trueの数を数えています）。


In [ ]:
# TODO: for a in [10, 100, 300, 1000, 3000]: で Lasso を作り、
#       外れ幅と、0になった列の数を表示する
# ヒント: l = Lasso(alpha=a, max_iter=100000).fit(Z_tr, y_tr)
...


α=10でも2本が0になり、α=1000では13本、α=3000では17本が0になりました。外れ幅は、α=1000で16,663ドルといちばん良く、α=3000では17,499ドルに悪くなりました。罰金を強くしすぎると、役に立つ列まで外してしまうのです。

どの列が、最初に0になったのでしょうか。列の名前と係数を組にして、係数が0の名前だけを残します。

```python
for c, v in zip(['a', 'b'], [1, 0]):
    print(c, v)       # a 1、b 0 と表示される
```

`zip`は、2つの並びを、同じ順に1組ずつ取り出します。`for c, v in`で、1組の中身を2つの名前に受け取ります。下のセルでは、列の名前`c`と係数`v`を組にして、`v == 0`の名前だけをリストに残しています。


In [ ]:
l10 = Lasso(alpha=10, max_iter=100000).fit(Z_tr, y_tr)
print([c for c, v in zip(num, l10.coef_) if v == 0])    # 0になった列


最初に0になったのは、2階の面積（`2nd Flr SF`）と、地下の未仕上げの面積（`Bsmt Unf SF`）でした。どちらも、1節で見た「足すとほかの列になる」組の一員です。居住面積と1階の面積がわかれば、2階の面積はほぼわかります。Lassoは、重なっている分を自分で見つけて外したのです。

係数の二乗の罰金（Ridge）は、係数が0に近づくほど、罰金の減り方がゆるやかになります（0.1² ＝ 0.01）。そのため、係数を0の手前まで縮めても、ちょうど0にする理由がありません。絶対値の罰金（Lasso）は、0の近くでも、係数を減らしたぶんだけ同じ割合で罰金が減ります（|0.1| ＝ 0.1）。役に立たない列は、係数を0まで減らしきったほうが得になるのです。


---


## 4. 罰金の強さを、交差検証で選ぶ


RidgeでもLassoでも、αで外れ幅と係数が変わりました。α=3000のLassoは、役に立つ列まで外しました。では、αはどう選べばよいでしょうか。

αは、強すぎても弱すぎてもいけません。テストは、最後まで取っておく本番でした（第2回）。テストの外れ幅を見てαを選ぶと、本番を覗いたことになります。そこで、訓練データの中で交差検証をしてαを選びます。`RidgeCV`と`LassoCV`は、αの候補ごとに交差検証をして、いちばん良いαを自動で選んでくれます。


### お手本：RidgeCVのalphasを読んでから使う

- 役割：`RidgeCV`の引数`alphas`の説明を読み、αの候補を渡して、交差検証でαを選ぶ
- 入力：`Z_tr`・`y_tr`
- 出力：選ばれたαと、テストの外れ幅

`np.logspace(-1, 4, 50)`は、0.1から10000までを、かけ算で等しい間隔に50個並べた候補です。`cv=5`は5分割の交差検証です。


In [ ]:
doc = RidgeCV.__doc__
i = doc.find('alphas :')
print(doc[i:i + 300])
rcv = RidgeCV(alphas=np.logspace(-1, 4, 50), cv=5).fit(Z_tr, y_tr)
print('選んだ α:', round(rcv.alpha_, 1))
print('外れ幅:', mae(y_te, rcv.predict(Z_te)), 'ドル')


### 変えてみる：LassoCVで選ぶ

`LassoCV`は、αの候補を自分で作ってくれます。`cv=5`は、RidgeCVと同じく5分割の交差検証です。


In [ ]:
lcv = LassoCV(cv=5, max_iter=100000).fit(Z_tr, y_tr)
coef_lasso = pd.Series(lcv.coef_, index=num)
print('選んだ α:', round(lcv.alpha_, 1))
print('外れ幅:', mae(y_te, lcv.predict(Z_te)), 'ドル')
print('0になった列:', coef_lasso[coef_lasso == 0].index.tolist())


交差検証で選ばれたαは、Ridgeが約36、Lassoが約193でした（3節で見たとおり、RidgeとLassoのαは物差しが違うので、大きさは比べません）。外れ幅は、正則化なしの16,970ドルから、Ridgeで16,861ドル、Lassoで16,841ドルと、わずかに良くなりました。

TODO③では、テストの外れ幅が最も良かったのはα=1000でした。交差検証が選んだα（約193）とは違いますが、それでかまいません。交差検証は訓練の中だけで選んでいて、テストの結果に合わせて選び直したら、本番を覗いたことになります。Lassoは、2階の面積・地下の未仕上げの面積・バスルームの数の3本を0にしました。

当たりは、ほとんど変わりません。今回のデータでは、正則化の効果は、当たりよりも、係数が説明に使える形になることにあります。

なお、ここでは標準化の物差しを訓練データ全体で決めてから交差検証をしています。厳密には、分けるたびに物差しを決め直すべきです（第3回の順番）。今日は、このまま進めます。


---


## 5. 説明に使える係数で、査定する


交差検証で選んだLassoは、重なる列を外しました。この係数で、お客さんへの説明を組み立てます。まず、Lassoの係数を、大きい順に見てみます。


In [ ]:
ranked = coef_lasso.sort_values(key=abs, ascending=False)
print(ranked.round(0).head(8))
print(coef_lasso[['Gr Liv Area', '1st Flr SF',
                  '2nd Flr SF']].round(0))


居住面積が1位、品質が2位です。この上位2つは、第4回の11本の式（標準化した係数）と同じです。3位以下には、地下の仕上げた部分の面積など、今日足した列も入ってきました。

5位の寝室の数（`Bedroom AbvGr`）は、1標準偏差（約0.8部屋）あたり−7,412ドルと、マイナスです。第4回のバスルームと同じく、「同じ広さなら、寝室が多いほど1部屋が狭い」差を映していると考えられます。「寝室が多いと安い」とは読みません。

そして、居住面積の効き目は、2階の面積を0にしたおかげで、居住面積の列にまとまっています。


### TODO④：係数を、元の単位に戻す

Lassoの係数は「1標準偏差（standard deviation）あたり」です。お客さんに伝えるには、「1㎡あたり」「1段階あたり」に戻す必要があります。係数を、標準化で割った標準偏差（`sc.scale_`）で割って元の単位に戻し、居住面積と品質の係数を、f-stringの文で表示してください。


In [ ]:
# TODO: coef_lasso / sc.scale_ で元の単位に戻し、
#       居住面積と品質の係数を文にする
# ヒント: raw_unit = coef_lasso / sc.scale_ のあと、
#         raw_unit['Gr Liv Area'] を f-string に埋め込む
...


居住面積の係数は1㎡あたり約662ドル、品質は1段階あたり約13,626ドルです。正則化なしの35本の式では4つの列に散らばっていた居住面積の効き目が、ほぼ一つの列にまとまりました。

ただし、1節で見たとおり、面積の係数は組み合わせて読みます。元の単位に戻した1階の係数は約39ドル、2階は0ドルです（`raw_unit[['Gr Liv Area', '1st Flr SF', '2nd Flr SF']]`で確かめられます）。2階の1㎡は居住面積の約662ドルだけですが、1階が1㎡広いと居住面積も1㎡広いので、1階の1㎡は662＋39＝約701ドルです。

査定担当としての答えです。35本の列を使って当たりを良くしても（外れ幅16,841ドル）、Lassoで係数を抑えれば、お客さんに説明できます。その説明は『2階の1㎡で約660ドル（1階は約700ドル）、品質1段階あたり約1万4千ドル』です。

まだ言えないことが、2つあります。

- 地区の違い：家の値段は場所で大きく違うはずだが、今の式には地区が入っていない
- 正則化した係数の揺れ：Lassoは、どの列を0にするかが、データが少し変わると入れ替わることがある（発展）


---


## 目標に答えられたか


- 今日の目標は「列を増やして当たりを良くしながら、説明できる係数を保つ」ことでした。35本の式の外れ幅と、Lassoの外れ幅、そして居住面積の係数はいくらでしたか？
- TODO①で、居住面積が3つの列の足し算だとわかりました。そのとき、係数が一つに決まらないのはなぜですか？
- Ridgeは、似た列の組の係数を、どう動かしましたか？Lassoはどうでしたか？
- αを、テストの外れ幅ではなく交差検証で選ぶのは、なぜですか？


---


## 今日の要点


- 列を増やすと当たりは良くなるが、足すとほかの列になる組があると、係数は一つに決まらない
- Ridgeは係数の二乗に罰金をかけ、似た列の組の係数を近づけながら縮める
- Lassoは係数の絶対値に罰金をかけ、いくつかの係数をちょうど0にする（列を選ぶ）
- 罰金の強さαは、テストではなく訓練の中の交差検証で選ぶ
- 今回のデータでは、正則化の効果は当たりよりも、係数が説明に使える形になること。元の単位に戻して伝える
- Python：関数の引数は、説明書き（`__doc__`・`help`）で確かめてから使う
- Python：リスト内包表記`[c for c in cols if 条件]`で、条件に合う列の名前を1行で選ぶ


---


## 次回へ


今日までの5回は、値段や台数のような数を当ててきました。

次回は、場面を健診の担当医に変えます。当てたいのは、「心臓病か、そうでないか」という区分です。数を当てる式を、どう区分の判定に変えるのか。そして、見逃しと誤検出は、どう数えればよいのでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

TODO②では、αを変えてRidgeを作りました。今度は、α=100のRidgeについて、車庫の台数（`Garage Cars`）と車庫の面積（`Garage Area`）の係数を、正則化なしの線形回帰（標準化したデータで立てたもの）の係数と並べて表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

正則化なしの線形回帰は`LinearRegression().fit(Z_tr, y_tr)`で、標準化したデータに立て直します（係数を同じ「1標準偏差あたり」にそろえるため）。Ridgeは`Ridge(alpha=100).fit(Z_tr, y_tr)`です。どちらも`.coef_[num.index('Garage Cars')]`で、車庫の台数の係数を取り出せます。

</details>


### 応用②（判断）

応用①で、α=100のRidgeでの、車庫の台数（`Garage Cars`）の係数は何ドルですか。四捨五入して整数（ドル）で答えてください（例: 1234）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`round(係数)`で整数にします。この係数は「車庫の台数が1標準偏差（約0.7台）増えたら、何ドル」という意味です。

</details>


### 応用③（解釈）

応用①では、係数を縮めるはずのRidgeで、車庫の台数と車庫の面積の係数が、どちらも正則化なしより大きくなりました。「なぜ大きくなったのか」と「お客さんに車庫の係数をどう伝えればよいか」を、同僚の査定担当者に向けて3行程度で説明してください。

応用①の表を、居住面積（`Gr Liv Area`）・品質（`Overall Qual`）・建築年（`Year Built`）の行にも広げて見るとよいでしょう。


（ここに3行程度で書く）


---


## 発展（任意）


### Lassoが選ぶ列は、揺れないのか

Lassoは、2階の面積を0にしました。でも、居住面積・1階・2階は「足すと等しい」関係なので、1階を0にしても同じように当てられるはずです。訓練データをブートストラップ（第4回の発展）で20回取り直し、そのたびにα=300のLassoで、どの列が0になるかを数えてみます。


In [ ]:
from collections import Counter
count = Counter()                 # 列ごとに、0になった回数を数える入れ物
for i in range(20):
    s = tr.sample(len(tr), replace=True, random_state=i)
    Zs = sc.transform(imp.transform(s[num]))
    l = Lasso(alpha=300, max_iter=100000).fit(Zs, s['SalePrice'])
    count.update([c for c, v in zip(num, l.coef_) if v == 0])
for c in ['1st Flr SF', '2nd Flr SF', 'Gr Liv Area', 'Bsmt Unf SF', 'Full Bath']:
    print(f'{c:>14}: 20回中 {count[c]:>2} 回、0になった')


読み方：20回の取り直しで、2階の面積と地下の未仕上げの面積は、20回とも0になりました。ところが、1階の面積も何回か0になり、バスルームの数は回によって0になったりならなかったりしています。完全に重なる組や、似た列がほかにある列では、Lassoがどれを0にするかは、データが少し変わると入れ替わることがあるのです。

つまり、「Lassoが2階の面積を0にした」ことは、「2階の広さは値段に影響しない」という意味ではありません。重なっている組の中で、どれか一つを代表に選んだだけです。Lassoの選んだ列を説明に使うときも、似た列の組はまとめて読むことが大切です。

試すなら、αを100や1000に変えて、0になる列の顔ぶれと、その安定ぐあいがどう変わるかを見てみましょう。
